In [1]:
import os
gpu_ids = [0]
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(map(str, gpu_ids))
import torch

In [2]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image, ImageColor

class ShirtColorDataset(Dataset):
    def __init__(self, csv_path, image_folder):
        self.df = pd.read_csv(csv_path)
        self.df = self.df[self.df['is_standalone'] == True]
        self.image_folder = image_folder
        self.transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
        ])
        self.valid_df = self.df[self.df['baseColour'].apply(self._is_valid_color)]

    def _is_valid_color(self, color):
        try:
            ImageColor.getrgb(color)
            return True
        except:
            return False

    def __len__(self):
        return len(self.valid_df)

    def __getitem__(self, idx):
        row = self.valid_df.iloc[idx]
        img_path = os.path.join(self.image_folder, f"{row['id']}.jpg")
        image = Image.open(img_path).convert("RGB")
        color_rgb = torch.tensor(ImageColor.getrgb(row['baseColour']), dtype=torch.float32) / 255.
        color_patch = color_rgb.view(3, 1, 1).expand(3, 256, 256)

        image_tensor = self.transform(image)
        input_tensor = torch.cat([image_tensor, color_patch], dim=0)
        return input_tensor, image_tensor  # 6ch input, 3ch target


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class ColorConditionedUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = UNetBlock(6, 64)
        self.enc2 = UNetBlock(64, 128)
        self.enc3 = UNetBlock(128, 256)
        self.pool = nn.MaxPool2d(2)
        self.up3 = UNetBlock(256, 128)
        self.up2 = UNetBlock(128, 64)
        self.final = nn.Conv2d(64, 3, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        d3 = self.up3(F.interpolate(e3, scale_factor=2))
        d2 = self.up2(F.interpolate(d3, scale_factor=2))
        return self.final(d2)


In [4]:
# train.py
import torch
from torch.utils.data import DataLoader
from torch import nn, optim
from tqdm import tqdm

def train(csv_path, image_folder, epochs=10, batch_size=8, lr=1e-4):
    dataset = ShirtColorDataset(csv_path, image_folder)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ColorConditionedUNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
        for x, y in progress_bar:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            progress_bar.set_postfix(loss=loss.item())
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss / len(dataloader):.4f}")

    torch.save(model.state_dict(), "color_unet.pth")
    print("Model saved to color_unet.pth")

# Example usage:



In [15]:
def change_shirt_color(image_path, mask_npy_path, target_color_name, save_path, blend_ratio=0.6):
    # Load image and resize
    image = Image.open(image_path).convert("RGB").resize((256, 256))
    image_np = np.array(image)

    # Load and resize mask
    mask_np = np.load(mask_npy_path)
    if mask_np.shape != (256, 256):
        mask_np = np.array(Image.fromarray(mask_np).resize((256, 256)))
    mask_binary = (mask_np > 0.5).astype(np.uint8)
    mask_binary = 1 - mask_binary  # invert if needed

    # Get target color
    try:
        target_rgb = ImageColor.getrgb(target_color_name)
    except:
        print(f"Invalid color name: {target_color_name}")
        return

    # Apply blending color tint
    new_image_np = image_np.copy()
    for c in range(3):
        original = new_image_np[:, :, c]
        tint = target_rgb[c]
        new_image_np[:, :, c] = (
            original * (1 - blend_ratio * mask_binary) +
            tint * blend_ratio * mask_binary
        )

    # Save output
    new_image = Image.fromarray(new_image_np.astype(np.uint8))
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    new_image.save(save_path)
    print(f"✅ Saved recolored image to {save_path}")


In [16]:
change_shirt_color(
    image_path="../fashion-dataset/images/4729.jpg",
    mask_npy_path="./cached_masks/4729.npy",
    target_color_name="Red",
    save_path="./recolored/4729_red.jpg"
)

✅ Saved recolored image to ./recolored/4729_red.jpg
